# DAMICORE: categorias de abuso

## Proposta de pesquisa

**Pergunta:** as categorias da fonte relacionadas à violência contra mulheres apresentam perfis contextuais recorrentes?

**Unidade de análise:** uma categoria da fonte, não uma pessoa ou uma denúncia. Todas as denúncias com vítima feminina entre janeiro de 2020 e junho de 2026 contribuem para o perfil agregado de cada categoria elegível em que aparecem.

**Escopo operacional:** categorias sob `INTEGRIDADE`, `VIDA` ou `VIOLÊNCIA INSTITUCIONAL`, além de `LIBERDADE > SEXUAL` e da categoria da fonte sobre violência política de gênero. Essa regra da taxonomia da fonte é exploratória, não uma definição jurídica de violência contra a mulher. O primeiro semestre de 2020 preserva rótulos legados sem a hierarquia `>`; esses rótulos ficam fora do subconjunto comparável até uma harmonização específica.

**Representação:** todas as categorias recebem o mesmo vocabulário ordenado de 20 dimensões: as 12 dimensões originais mais início das violações, motivação, raça/cor, escolaridade e renda da vítima, etnia da vítima, faixa etária do suspeito e escolaridade do suspeito. Cada valor é serializado como desvio relativo à distribuição global, em largura fixa de `00000` a `10000`; zero relativo significa exatamente a prevalência global. O suporte total da categoria fica fora do documento comprimido.

**Hipótese exploratória:** categorias com desvios relativos semelhantes de perfil demográfico, relação, ambiente, registro, tempo, motivação e características do suspeito terão menores distâncias NCD, aparecerão próximas na árvore DAMICORE e tenderão a compartilhar um cluster.

A cadeia do experimento é: `desvio contextual relativo → matriz de distâncias NCD → diagnóstico de viés → árvore → clusters DAMICORE → visualizações quantitativas`. A proposta é uma comparação exploratória de perfis contextuais; ela não confirma tipologias jurídicas.


In [ ]:
from datetime import datetime
from pathlib import Path
from itertools import combinations
from math import log10, log2
import json
import os
import sys
import textwrap

import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import pandas as pd
import psycopg
from damicore import estimate, run
from dotenv import load_dotenv
from IPython.display import display
import toytree
import toyplot

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Execute este notebook a partir da raiz do repositório ou do diretório notebooks.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from scripts.damicore_case_experiment import CASE_FIELDS

load_dotenv(PROJECT_ROOT / ".env")
DATABASE_URL = os.getenv("DISQUE100_DATABASE_URL", "postgresql://postgres@127.0.0.1:5433/disque100")
START_DATE = "2020-01-01"
END_DATE = "2026-07-01"
VICTIM_GENDER = "FEMININO"
MIN_REPORT_COUNT = 50
SMOOTHING_ALPHA = 20
LOG_RATIO_LIMIT = 4.0
BIAS_WARNING_THRESHOLD = 0.70

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "damicore_abuse_categories_2020_2026"
WORK_DIR = ARTIFACT_DIR / "work"
CORPUS_DIR = WORK_DIR / "normalized-corpus"
RESULTS_DIR = ARTIFACT_DIR / "results"
CORPUS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def wrapped_label(text, width=30):
    return '\n'.join(textwrap.wrap(text, width=width))

VISUAL_STYLE = {
    'font_size': 7,
    'text_color': '#263238',
    'grid_color': '#E2E7E9',
    'accent_color': '#2A6F97',
    'accent_light': '#CFE4EE',
}


## 1. Cobertura, escopo e contextos agregados

`source_hash` é usado somente dentro do PostgreSQL para contar denúncias distintas. O notebook recebe totais de cobertura e contagens agregadas, nunca denúncias individuais. Valores nulos, vazios ou apenas espaços nas 20 dimensões recebem `DESCONHECIDO` para que toda denúncia permaneça representada.


In [ ]:
category_sql = '''
nullif(concat_ws(' > ',
    nullif(trim(split_part(violation, '>', 1)), ''),
    nullif(trim(split_part(violation, '>', 2)), '')
), '')
'''

coverage_query = f'''
WITH reports AS (
    SELECT source_hash,
           bool_or(violation IS NOT NULL) AS has_violation,
           bool_or(({category_sql}) IS NOT NULL) AS has_category
    FROM public.disque100_reports
    WHERE registered_at >= %s AND registered_at < %s
      AND victim_gender = %s
    GROUP BY source_hash
)
SELECT count(*) AS female_reports,
       count(*) FILTER (WHERE has_category) AS reports_with_category,
       count(*) FILTER (WHERE NOT has_violation) AS reports_without_violation
FROM reports
'''

context_query = f'''
WITH report_values AS (
    SELECT DISTINCT
        source_hash,
        {category_sql} AS category,
        coalesce(nullif(trim(victim_age_group), ''), 'DESCONHECIDO') AS age_group,
        coalesce(nullif(trim(victim_suspect_relationship), ''), 'DESCONHECIDO') AS relationship,
        coalesce(nullif(trim(violation_setting), ''), 'DESCONHECIDO') AS setting,
        to_char(date_trunc('month', registered_at), 'YYYY-MM') AS month,
        coalesce(nullif(trim(violation_start_period), ''), 'DESCONHECIDO') AS violation_start_period,
        coalesce(nullif(trim(service_channel), ''), 'DESCONHECIDO') AS service_channel,
        coalesce(nullif(trim(reporter_type), ''), 'DESCONHECIDO') AS reporter_type,
        coalesce(nullif(trim(frequency), ''), 'DESCONHECIDO') AS frequency,
        coalesce(nullif(trim(emergency_status), ''), 'DESCONHECIDO') AS emergency_status,
        coalesce(nullif(trim(motivation), ''), 'DESCONHECIDO') AS motivation,
        coalesce(nullif(trim(vulnerable_group), ''), 'DESCONHECIDO') AS vulnerable_group,
        coalesce(nullif(trim(victim_disability), ''), 'DESCONHECIDO') AS victim_disability,
        coalesce(nullif(trim(victim_race_color), ''), 'DESCONHECIDO') AS victim_race_color,
        coalesce(nullif(trim(victim_education_level), ''), 'DESCONHECIDO') AS victim_education_level,
        coalesce(nullif(trim(victim_income_range), ''), 'DESCONHECIDO') AS victim_income_range,
        coalesce(nullif(trim(victim_ethnicity), ''), 'DESCONHECIDO') AS victim_ethnicity,
        coalesce(nullif(trim(suspect_age_group), ''), 'DESCONHECIDO') AS suspect_age_group,
        coalesce(nullif(trim(suspect_gender), ''), 'DESCONHECIDO') AS suspect_gender,
        coalesce(nullif(trim(suspect_education_level), ''), 'DESCONHECIDO') AS suspect_education_level,
        coalesce(nullif(trim(suspect_legal_nature), ''), 'DESCONHECIDO') AS suspect_legal_nature
    FROM public.disque100_reports
    WHERE registered_at >= %s AND registered_at < %s
      AND victim_gender = %s
), contexts AS (
    SELECT category, source_hash, 'denuncia'::text AS dimension, 'TODAS'::text AS value
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'faixa_etaria', age_group
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'relacao_vitima_suspeito', relationship
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'ambiente', setting
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'mes', month
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'inicio_violacoes', violation_start_period
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'canal_atendimento', service_channel
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'tipo_denunciante', reporter_type
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'frequencia', frequency
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'situacao_emergencia', emergency_status
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'motivacao', motivation
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'grupo_vulneravel', vulnerable_group
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'deficiencia_vitima', victim_disability
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'raca_cor_vitima', victim_race_color
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'escolaridade_vitima', victim_education_level
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'renda_vitima', victim_income_range
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'etnia_vitima', victim_ethnicity
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'faixa_etaria_suspeito', suspect_age_group
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'genero_suspeito', suspect_gender
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'escolaridade_suspeito', suspect_education_level
    FROM report_values WHERE category IS NOT NULL
    UNION ALL
    SELECT category, source_hash, 'natureza_juridica_suspeito', suspect_legal_nature
    FROM report_values WHERE category IS NOT NULL
)
SELECT category, dimension, value, count(DISTINCT source_hash) AS report_count
FROM contexts
GROUP BY category, dimension, value
ORDER BY category, dimension, value
'''

parameters = (START_DATE, END_DATE, VICTIM_GENDER)
with psycopg.connect(DATABASE_URL) as connection:
    with connection.cursor() as cursor:
        cursor.execute(coverage_query, parameters)
        coverage = pd.DataFrame(cursor.fetchall(), columns=[column.name for column in cursor.description])
        cursor.execute(context_query, parameters)
        context_counts = pd.DataFrame(cursor.fetchall(), columns=[column.name for column in cursor.description])


### Context Counts Overview

`context_counts` is the long-form aggregate table with columns `category`, `dimension`, `value`, `report_count`. `report_count` counts distinct `source_hash`.

In [ ]:
import numpy as np

# Assertions
assert set(context_counts.columns) == {'category', 'dimension', 'value', 'report_count'}
assert 'denuncia' in context_counts['dimension'].values

# Compact category summary
category_summary = context_counts.loc[
    context_counts['dimension'] == 'denuncia', ['category', 'report_count']
].sort_values('report_count', ascending=False)
print(f"SQL result: {len(context_counts):,} category-context rows")
display(category_summary.head(50))

# Display complete table if reasonably sized
if len(context_counts) < 1000:
    display(context_counts)
else:
    print(f"Table too large to display entirely ({len(context_counts)} rows), showing head:")
    display(context_counts.head())

# Keep the heatmap focused on the categories that will become DAMICORE objects.
selected_categories = category_summary.loc[
    (
        category_summary['category'].str.startswith(('INTEGRIDADE', 'VIDA', 'VIOLÊNCIA INSTITUCIONAL'))
        | category_summary['category'].eq('LIBERDADE > SEXUAL')
        | category_summary['category'].str.contains(
            'VIOLÊNCIA POLITÍCA DE GÊNERO E CONTRA AS MULHERES', regex=False
        )
    )
    & category_summary['report_count'].ge(MIN_REPORT_COUNT),
    'category',
].tolist()
assert selected_categories
heatmap_source = context_counts.loc[
    context_counts['category'].isin(selected_categories)
    & context_counts['dimension'].ne('denuncia')
].copy()
assert heatmap_source['dimension'].nunique() > 0
# The heatmap source should be raw context_counts as requested.

dimensions = sorted(heatmap_source['dimension'].unique())
categories = sorted(selected_categories)
global_log_vmax = max(1.0, float(np.log1p(heatmap_source['report_count'].max())))

figure, axes = plt.subplots(
    1, len(dimensions),
    figsize=(max(18, 2.6 * len(dimensions)), max(6, 0.42 * len(categories))),
    sharey=True, squeeze=False, constrained_layout=True,
)
axes = axes[0]
image = None

for ax, dim in zip(axes, dimensions):
    dim_data = heatmap_source[heatmap_source['dimension'] == dim]
    pivot_df = dim_data.pivot(index='category', columns='value', values='report_count').fillna(0)
    pivot_df = pivot_df.reindex(categories, fill_value=0)

    image = ax.imshow(
        np.log1p(pivot_df.to_numpy()), aspect='auto', cmap='YlGnBu',
        vmin=0, vmax=global_log_vmax,
    )
    ax.set_title(dim)
    ax.set_xticks(np.arange(len(pivot_df.columns)))
    ax.set_xticklabels(pivot_df.columns, rotation=90, fontsize=6)
    ax.tick_params(axis='x', length=0)
    if ax is axes[0]:
        ax.set_yticks(np.arange(len(categories)))
        ax.set_yticklabels([wrapped_label(category, width=28) for category in categories], fontsize=7)
    else:
        ax.tick_params(axis='y', length=0)


figure.suptitle('Distinct report counts by eligible category and context', ha='left', x=0.01)
figure.colorbar(image, ax=axes.tolist(), label='log1p(distinct report count)', shrink=0.85)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
figure.savefig(RESULTS_DIR / 'category-context-counts-heatmap.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:

display(coverage.rename(columns={
    "female_reports": "Denúncias com vítima feminina",
    "reports_with_category": "Denúncias com categoria",
    "reports_without_violation": "Denúncias sem violação",
}))
expected_dimension_labels = {label for _, label in CASE_FIELDS}
actual_dimension_labels = set(context_counts["dimension"].unique()) - {"denuncia"}
assert len(expected_dimension_labels) == 20
assert actual_dimension_labels == expected_dimension_labels
dimension_summary = (
    context_counts.loc[context_counts["dimension"].ne("denuncia")]
    .groupby("dimension", as_index=False)
    .agg(category_count=("category", "nunique"), value_count=("value", "nunique"))
    .sort_values("dimension")
)
display(dimension_summary)

category_support = (
    context_counts.loc[context_counts["dimension"] == "denuncia", ["category", "report_count"]]
    .rename(columns={"report_count": "category_report_count"})
    .sort_values("category")
)

scope_mask = (
    category_support["category"].str.startswith(("INTEGRIDADE", "VIDA", "VIOLÊNCIA INSTITUCIONAL"))
    | category_support["category"].eq("LIBERDADE > SEXUAL")
    | category_support["category"].str.contains(
        "VIOLÊNCIA POLITÍCA DE GÊNERO E CONTRA AS MULHERES", regex=False
    )
)
category_support["status"] = "incluída"
category_support.loc[~scope_mask, "status"] = "fora do escopo operacional"
category_support.loc[
    scope_mask & (category_support["category_report_count"] < MIN_REPORT_COUNT), "status"
] = "abaixo do suporte mínimo"

included_support = category_support.loc[category_support["status"] == "incluída"].copy()
excluded_categories = category_support.loc[category_support["status"] != "incluída"].copy()
included_categories = included_support["category"].tolist()
included_dimension_coverage = (
    context_counts.loc[
        context_counts["category"].isin(included_categories)
        & context_counts["dimension"].ne("denuncia")
    ]
    .groupby("dimension")["category"]
    .nunique()
)
assert set(included_dimension_coverage.index) == expected_dimension_labels
assert included_dimension_coverage.eq(len(included_categories)).all()
display(included_dimension_coverage.rename("eligible_category_count").to_frame())


print(f"Categorias da fonte: {len(category_support)}")
print(f"Categorias incluídas no DAMICORE: {len(included_support)}")
print(f"Suporte mínimo: {MIN_REPORT_COUNT} denúncias distintas")
display(included_support)
display(excluded_categories)


support_plot = included_support.sort_values("category_report_count")
figure, axis = plt.subplots(
    figsize=(12, max(6, 0.42 * len(support_plot))),
)
axis.barh(
    [wrapped_label(category, width=38) for category in support_plot["category"]],
    support_plot["category_report_count"],
    color=VISUAL_STYLE["accent_color"],
)
axis.set_xscale("log")
axis.set_xlabel("Distinct reports (log scale)")
axis.set_title("Support of eligible categories", loc="left", weight="bold")
axis.grid(axis="x", color=VISUAL_STYLE["grid_color"], linewidth=0.7)
axis.set_axisbelow(True)
axis.set_xlim(1, support_plot["category_report_count"].max() * 2)
for position, value in enumerate(support_plot["category_report_count"]):
    axis.text(
        value * 1.08,
        position,
        f"{int(value):,}",
        va="center",
        fontsize=8,
        color=VISUAL_STYLE["text_color"],
    )
figure.tight_layout()
figure.savefig(
    RESULTS_DIR / "category-support.png",
    dpi=180,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()


## 2. Documento normalizado de largura fixa por categoria elegível

O total da categoria é usado como denominador. O documento agora usa 20 dimensões, com `mes` representando o registro e `inicio_violacoes` representando o período informado para o início da violência. Para reduzir o ruído de categorias raras, usamos suavização empírica com `alpha=20`: `(contagem + alpha * prevalência_global) / (suporte + alpha)`. Em seguida, calculamos `log2(prevalência_suavizada / prevalência_global)`. Assim, zero representa o padrão global, valores positivos indicam sobre-representação e valores negativos indicam sub-representação. O resultado é codificado em largura fixa (`00000` a `10000`) para todas as categorias.


In [ ]:
for old_file in CORPUS_DIR.glob("*.txt"):
    old_file.unlink()

profile_counts = context_counts.loc[
    context_counts["category"].isin(included_categories)
    & context_counts["dimension"].ne("denuncia")
].copy()
vocabulary = (
    profile_counts[["dimension", "value"]]
    .drop_duplicates()
    .sort_values(["dimension", "value"])
)
assert set(vocabulary["dimension"]) == expected_dimension_labels
assert len(vocabulary["dimension"].unique()) == 20

normalized_profiles = (
    included_support[["category", "category_report_count"]]
    .merge(vocabulary, how="cross")
    .merge(profile_counts, on=["category", "dimension", "value"], how="left")
    .sort_values(["category", "dimension", "value"])
)
assert normalized_profiles.groupby("category").size().nunique() == 1
normalized_profiles["report_count"] = normalized_profiles["report_count"].fillna(0).astype(int)
global_counts = (
    profile_counts.groupby(["dimension", "value"], as_index=False)["report_count"]
    .sum()
    .rename(columns={"report_count": "global_count"})
)
global_counts["global_prevalence"] = (
    global_counts["global_count"]
    / global_counts.groupby("dimension")["global_count"].transform("sum")
)
normalized_profiles = normalized_profiles.merge(
    global_counts[["dimension", "value", "global_prevalence"]],
    on=["dimension", "value"],
    how="left",
    validate="many_to_one",
)
normalized_profiles["smoothed_prevalence"] = (
    normalized_profiles["report_count"]
    + SMOOTHING_ALPHA * normalized_profiles["global_prevalence"]
) / (normalized_profiles["category_report_count"] + SMOOTHING_ALPHA)
normalized_profiles["relative_log2"] = (
    normalized_profiles["smoothed_prevalence"]
    / normalized_profiles["global_prevalence"]
).map(log2)
normalized_profiles["relative_bps"] = (
    10_000
    * (normalized_profiles["relative_log2"].clip(-LOG_RATIO_LIMIT, LOG_RATIO_LIMIT) + LOG_RATIO_LIMIT)
    / (2 * LOG_RATIO_LIMIT)
).round().astype(int)

category_rows = []
for number, category in enumerate(sorted(included_categories), start=1):
    label = f"category-{number:03d}.txt"
    rows = normalized_profiles.loc[normalized_profiles["category"] == category]
    lines = [
        f"{row.dimension}|{row.value}|{row.relative_bps:05d}"
        for row in rows.itertuples(index=False)
    ]
    (CORPUS_DIR / label).write_text("\n".join(lines) + "\n", encoding="utf-8")
    support = int(rows["category_report_count"].iloc[0])
    category_rows.append({"label": label, "category": category, "report_count": support})

category_map = pd.DataFrame(category_rows)
category_map.to_csv(WORK_DIR / "normalized-category-map.csv", index=False)

corpus_sizes = pd.Series(
    {(path.name): path.stat().st_size for path in CORPUS_DIR.glob("*.txt")},
    name="bytes",
)
assert corpus_sizes.nunique() == 1, "Os documentos normalizados devem ter o mesmo tamanho em bytes."
display(category_map)
print(f"Dimensões contextuais: {len(vocabulary['dimension'].unique())}")
print(f"Chaves contextuais por categoria: {len(vocabulary)}")
print(f"Bytes por documento normalizado: {int(corpus_sizes.iloc[0])}")
print(f"Suavização empírica alpha: {SMOOTHING_ALPHA}")
print(f"Limite do log-ratio: ±{LOG_RATIO_LIMIT}")


## 3. Fluxo mínimo do DAMICORE

A sequência é `estimate → run → membership → matriz de distâncias → árvore`. Valores NCD menores significam perfis contextuais normalizados mais semelhantes. Cada execução recebe automaticamente um diretório com data, hora e microssegundos; por isso, uma execução nova nunca colide com uma execução anterior.


In [ ]:
preview = estimate(CORPUS_DIR, source_kind="files")
display(preview.model_dump())
if not preview.within_limits:
    raise RuntimeError(f"Os limites do DAMICORE foram excedidos: {preview.violations}")

RUN_ID = datetime.now().astimezone().strftime("%Y%m%d-%H%M%S-%f")
RUN_DIR = WORK_DIR / f"normalized-run-{RUN_ID}"
print(f"Identificador automático desta execução: {RUN_ID}")
result = run(CORPUS_DIR, source_kind="files", output_dir=RUN_DIR)
try:
    membership = result.membership.copy()
    distance_matrix = result.distance_matrix.to_pandas()
    tree_newick = result.tree_newick
finally:
    result.close()

category_clusters = (
    membership[["object_id", "label", "cluster"]]
    .merge(category_map, on="label", validate="one_to_one")
    .sort_values(["cluster", "category"])
)
category_clusters[["category", "cluster", "report_count"]].to_csv(
    RESULTS_DIR / "category-clusters.csv", index=False
)

label_to_category = category_map.set_index("label")["category"].to_dict()
category_distance_matrix = distance_matrix.rename(
    index=label_to_category, columns=label_to_category
)
clusters = sorted(category_clusters["cluster"].unique())
cluster_colors = {
    cluster: plt.colormaps["tab20"](index % 20)
    for index, cluster in enumerate(clusters)
}
cluster_hex_colors = {cluster: to_hex(color) for cluster, color in cluster_colors.items()}

display(category_clusters[["category", "cluster", "report_count"]])
display(category_distance_matrix.round(3))


## 4. Verificação de predominância do tamanho ou da popularidade

Estes são diagnósticos de triagem, não testes de significância. Uma correlação absoluta alta (sinalizada aqui a partir de 0,70) significa que a NCD ainda acompanha fortemente o suporte ou o tamanho bruto do documento; nesse caso, os clusters não devem ser interpretados como perfis contextuais.


In [ ]:
category_lookup = category_clusters.set_index("label")
pair_rows = []
for left_label, right_label in combinations(distance_matrix.index, 2):
    left = category_lookup.loc[left_label]
    right = category_lookup.loc[right_label]
    pair_rows.append({
        "ncd": float(distance_matrix.loc[left_label, right_label]),
        "support_gap": abs(log10(left["report_count"]) - log10(right["report_count"])),
        "byte_gap": abs(corpus_sizes[left_label] - corpus_sizes[right_label]),
    })

pair_diagnostics = pd.DataFrame(pair_rows)
ncd_rank = pair_diagnostics["ncd"].rank()
support_correlation = ncd_rank.corr(pair_diagnostics["support_gap"].rank())
byte_correlation = ncd_rank.corr(pair_diagnostics["byte_gap"].rank())
diagnostic_summary = pd.DataFrame({
    "diagnostic": ["NCD versus diferença absoluta de suporte em log", "NCD versus diferença de bytes do corpus"],
    "spearman_correlation": [support_correlation, byte_correlation],
})
display(diagnostic_summary.round(3))

if pd.notna(support_correlation) and abs(support_correlation) >= BIAS_WARNING_THRESHOLD:
    print("AVISO: a popularidade das categorias ainda influencia fortemente a NCD; não interprete os clusters.")
else:
    print("A correlação com a diferença de suporte está abaixo do limite de triagem.")
if pd.isna(byte_correlation):
    print("Todos os documentos normalizados têm o mesmo tamanho em bytes; a correlação com a diferença de bytes é indefinida, como esperado.")
elif abs(byte_correlation) >= BIAS_WARNING_THRESHOLD:
    print("AVISO: o tamanho do documento ainda influencia fortemente a NCD; não interprete os clusters.")
else:
    print("A correlação com a diferença de bytes está abaixo do limite de triagem.")


## 5. Matriz de distância NCD

A matriz mostra as distâncias entre as categorias elegíveis. Valores menores indicam mais estrutura compressível compartilhada. A ordem dos eixos segue os clusters DAMICORE para facilitar a comparação visual.


In [ ]:
ncd_order = (
    category_clusters.sort_values(["cluster", "category"])["category"].tolist()
)
ncd_matrix = category_distance_matrix.loc[ncd_order, ncd_order]

figure, axis = plt.subplots(
    figsize=(max(10, 0.68 * len(ncd_order)), max(8, 0.58 * len(ncd_order))),
)
image = axis.imshow(
    ncd_matrix.to_numpy(),
    cmap="Blues",
    aspect="equal",
    vmin=0,
    vmax=float(np.nanmax(ncd_matrix.to_numpy())),
)
labels = [wrapped_label(category, width=24) for category in ncd_order]
axis.set_xticks(np.arange(len(labels)))
axis.set_xticklabels(labels, rotation=90, fontsize=7)
axis.set_yticks(np.arange(len(labels)))
axis.set_yticklabels(labels, fontsize=7)
axis.set_xlabel("Category ordered by DAMICORE cluster")
axis.set_ylabel("Category ordered by DAMICORE cluster")
axis.set_title("NCD distances between eligible categories", loc="left", weight="bold")
axis.grid(False)

cluster_boundaries = (
    category_clusters.groupby("cluster", sort=True).size().cumsum().tolist()
)
for boundary in cluster_boundaries[:-1]:
    axis.axhline(boundary - 0.5, color=VISUAL_STYLE["accent_color"], linewidth=1.0)
    axis.axvline(boundary - 0.5, color=VISUAL_STYLE["accent_color"], linewidth=1.0)

figure.colorbar(image, ax=axis, label="NCD distance", shrink=0.82)
figure.tight_layout()
figure.savefig(
    RESULTS_DIR / "category-ncd-heatmap.png",
    dpi=180,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()


## 6. Árvore de distância do DAMICORE

As folhas mostram os nomes reais das categorias. Categorias com caminhos de ligação menores têm desvios relativos de contexto mais semelhantes. A cor identifica o cluster DAMICORE; ela não representa uma classificação jurídica.


In [ ]:
tree = toytree.tree(tree_newick)
by_object_id = category_clusters.set_index("object_id")
by_label = category_clusters.set_index("label")
tip_labels = []
tip_colors = []

for tip_name in tree.get_tip_labels():
    if tip_name in by_object_id.index:
        category_row = by_object_id.loc[tip_name]
    elif tip_name in by_label.index:
        category_row = by_label.loc[tip_name]
    else:
        raise KeyError(f"Folha desconhecida na árvore DAMICORE: {tip_name}")
    cluster = int(category_row["cluster"])
    tip_labels.append(category_row["category"])
    tip_colors.append(cluster_hex_colors[cluster])

if len(tip_labels) != len(category_clusters):
    raise ValueError("A árvore DAMICORE não corresponde às categorias esperadas.")

canvas, axes, mark = tree.draw(
    layout="r",
    width=1500,
    height=max(480, len(category_clusters) * 42),
    tip_labels=tip_labels,
    tip_labels_colors=tip_colors,
    tip_labels_align=True,
    tip_labels_style={"font-size": "12px", "font-family": "Arial"},
    node_labels=False,
    node_mask=False,
    edge_style={
        "stroke": "#607078",
        "stroke-width": 1.5,
        "stroke-opacity": 0.85,
    },
    scale_bar=True,
)
canvas.text(35, 24, "Árvore DAMICORE das categorias de abuso", style={"font-size": "19px", "font-weight": "bold", "fill": "#263238"})
canvas.text(35, 46, "Categorias próximas têm perfis normalizados mais semelhantes nas 20 dimensões contextuais", style={"font-size": "12px", "fill": "#607078"})
legend_markers = [
    (
        f"Grupo {cluster}",
        toyplot.marker.create(
            shape="s",
            size=12,
            mstyle={"fill": cluster_hex_colors[cluster], "stroke": cluster_hex_colors[cluster]},
        ),
    )
    for cluster in clusters
]
canvas.legend(
    legend_markers,
    corner=("top-right", 24, 210, max(72, 28 * len(clusters) + 24)),
    label="Grupo DAMICORE",
)
toytree.save(canvas, RESULTS_DIR / "category-tree.html")
toytree.save(canvas, RESULTS_DIR / "category-tree.svg")
canvas


## Limite de interpretação

O DAMICORE agrupa categorias elegíveis da fonte cujos desvios relativos de contexto demográfico, do processo de registro, da relação, do ambiente, do mês e das demais 20 dimensões são comprimidos de forma semelhante. Interprete a árvore e os agrupamentos somente quando o diagnóstico de viés for aceitável. A motivação pode aproximar categorias por conteúdo semântico, enquanto os campos socioeconômicos e étnicos podem ter ausência elevada. Mesmo assim, os clusters são perfis contextuais exploratórios, não tipologias jurídicas ou sociais confirmadas de abuso; as categorias esparsas excluídas precisam de uma análise descritiva separada.


## 7. Segundo experimento — corpus de denúncias completas

O experimento normalized acima permanece como a abordagem agregada de referência, agora ampliada para as mesmas 20 dimensões usadas nos casos. Esta seção cria um arquivo por categoria com uma linha canônica para cada source_hash distinto da categoria. A linha mantém os valores das 20 dimensões juntos dentro da denúncia.

Serão comparados dois regimes: case-full, com todos os casos distintos, e case-balanced, com o mesmo número de casos por categoria. O balanceamento é uniforme por source_hash, sem reposição dentro de uma réplica; combinações frequentes continuam mais frequentes porque correspondem a mais denúncias distintas.


In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.damicore_case_experiment import (
    CASE_COLUMNS,
    CASE_FIELDS,
    build_case_corpora,
    compare_combination_distributions,
    load_case_records,
)
assert len(CASE_FIELDS) == 20
assert len(CASE_COLUMNS) == 20
assert {label for _, label in CASE_FIELDS} == expected_dimension_labels


CASE_SEEDS = [101, 202, 303, 404, 505]
CASE_WORK_DIR = WORK_DIR / "case-corpus"
CASE_CATEGORY_MAP_PATH = CASE_WORK_DIR / "case-category-map.csv"
CASE_COMBINATION_COUNTS_PATH = CASE_WORK_DIR / "case-combination-counts.csv"
CASE_COMBINATION_COMPARISON_PATH = CASE_WORK_DIR / "case-combination-comparison.csv"

normalized_distance_matrix = distance_matrix.copy()
normalized_membership = membership.copy()
normalized_category_clusters = category_clusters.copy()
normalized_corpus_sizes = corpus_sizes.copy()

case_records = load_case_records(
    database_url=DATABASE_URL,
    start_date=START_DATE,
    end_date=END_DATE,
    victim_gender=VICTIM_GENDER,
    included_categories=included_categories,
    category_sql=category_sql,
)

case_support = (
    case_records.groupby("category", as_index=False)["source_hash"]
    .nunique()
    .rename(columns={"source_hash": "case_count"})
)
expected_support = included_support[["category", "category_report_count"]].rename(
    columns={"category_report_count": "expected_case_count"}
)
case_support_check = expected_support.merge(
    case_support, on="category", validate="one_to_one"
)
assert (
    case_support_check["expected_case_count"]
    == case_support_check["case_count"]
).all()

case_category_map, case_combination_counts, balanced_sample_size = build_case_corpora(
    case_records=case_records,
    category_map=category_map,
    work_dir=WORK_DIR,
    seeds=CASE_SEEDS,
)
case_combination_comparison = compare_combination_distributions(
    case_combination_counts,
    CASE_COMBINATION_COMPARISON_PATH,
)

print(f"Casos categoria-relatório: {len(case_records):,}")
print(f"Tamanho balanceado por réplica: {balanced_sample_size:,}")
print(f"Réplicas balanceadas: {len(CASE_SEEDS)}")
display(case_support.sort_values("case_count", ascending=False))
display(
    case_records[["category", "canonical_record"]].head(3)
)


### 7.1 Construção dos documentos de casos

O agrupamento é feito no grão source_hash + category. Um mesmo relatório multilabel pode aparecer uma vez em mais de um arquivo de categoria, mas nunca mais de uma vez na mesma categoria. Os identificadores permanecem apenas na memória para validar o grão e não são gravados nos artefatos. Os campos com múltiplos valores são deduplicados, ordenados e serializados juntos.


In [ ]:
print("Corpus completo:")
display(
    case_category_map.loc[
        case_category_map["regime"] == "case-full",
        ["label", "category", "case_count", "bytes"],
    ].sort_values("category")
)

print("Corpus balanceado:")
display(
    case_category_map.loc[
        case_category_map["regime"] == "case-balanced",
        ["replicate", "category", "case_count", "bytes"],
    ].sort_values(["replicate", "category"]).head(28)
)

assert case_category_map["case_count"].gt(0).all()
assert set(case_category_map["category"]) == set(included_categories)
assert case_category_map.loc[
    case_category_map["regime"] == "case-balanced", "case_count"
].eq(balanced_sample_size).all()
assert not case_records.duplicated(["source_hash", "category"]).any()
canonical_keys = case_records["canonical_record"].map(lambda value: set(json.loads(value)))
assert canonical_keys.map(lambda keys: not {"source_hash", "category", "id"}.intersection(keys)).all()
assert canonical_keys.map(lambda keys: keys == expected_dimension_labels).all()
assert case_combination_counts["case_count"].gt(0).all()
assert set(case_combination_counts["category"]) == set(included_categories)


### 7.2 Execução dos regimes de casos

case-full mede similaridade contextual junto com prevalência e volume. Cada réplica case-balanced mede a estrutura das combinações sob suporte igual. O DAMICORE recebe os arquivos de linhas canônicas; as contagens explícitas ficam no CSV auxiliar.


In [ ]:
def run_case_damicore(experiment_name, corpus_dir):
    preview = estimate(corpus_dir, source_kind="files")
    preview_data = preview.model_dump()
    preview_data["experiment"] = experiment_name
    print(
        f"{experiment_name}: "
        f"{preview.object_count} objects, "
        f"{preview.pair_count} pairs, "
        f"{preview.input_size_bytes:,} input bytes"
    )

    if not preview.within_limits:
        preview_data["status"] = "resource_limit"
        return {
            "name": experiment_name,
            "preview": preview_data,
            "status": "resource_limit",
            "corpus_dir": corpus_dir,
        }

    run_id = datetime.now().astimezone().strftime("%Y%m%d-%H%M%S-%f")
    output_dir = CASE_WORK_DIR / f"{experiment_name}-run-{run_id}"
    result = run(corpus_dir, source_kind="files", output_dir=output_dir)
    try:
        result_data = {
            "membership": result.membership.copy(),
            "distance_matrix": result.distance_matrix.to_pandas(),
            "tree_newick": result.tree_newick,
        }
    finally:
        result.close()

    preview_data["status"] = "completed"
    result_data.update(
        {
            "name": experiment_name,
            "preview": preview_data,
            "status": "completed",
            "corpus_dir": corpus_dir,
            "output_dir": output_dir,
        }
    )
    return result_data

case_results = {}
case_results["case-full"] = run_case_damicore(
    "case-full",
    CASE_WORK_DIR / "full",
)
for replicate in [f"replicate-{index:03d}" for index in range(1, len(CASE_SEEDS) + 1)]:
    experiment_name = f"case-balanced-{replicate}"
    case_results[experiment_name] = run_case_damicore(
        experiment_name,
        CASE_WORK_DIR / "balanced" / replicate,
    )

case_run_summary = pd.DataFrame(
    [
        {
            "experiment": name,
            "status": data["status"],
            "object_count": data["preview"].get("object_count"),
            "pair_count": data["preview"].get("pair_count"),
            "input_size_bytes": data["preview"].get("input_size_bytes"),
            "output_dir": str(data.get("output_dir", "")),
        }
        for name, data in case_results.items()
    ]
)
case_run_summary.to_csv(CASE_WORK_DIR / "case-run-summary.csv", index=False)
display(case_run_summary)


### 7.3 Comparação entre as abordagens

A comparação separa similaridade contextual, efeito do suporte e estabilidade das partições. Se case-full e case-balanced forem semelhantes, o volume não controla sozinho o agrupamento. Se divergirem, essa divergência mostra que a prevalência participa da estrutura comprimida.


In [ ]:
def category_named_distance(matrix):
    label_to_category = category_map.set_index("label")["category"].to_dict()
    renamed = matrix.rename(index=label_to_category, columns=label_to_category)
    return renamed.loc[included_categories, included_categories]

distance_matrices = {
    "normalized": category_named_distance(normalized_distance_matrix)
}
for name, data in case_results.items():
    if data["status"] == "completed":
        distance_matrices[name] = category_named_distance(data["distance_matrix"])

distance_comparison_rows = []
for left_name, right_name in combinations(distance_matrices, 2):
    left_values = []
    right_values = []
    for left_category, right_category in combinations(included_categories, 2):
        left_values.append(
            distance_matrices[left_name].loc[left_category, right_category]
        )
        right_values.append(
            distance_matrices[right_name].loc[left_category, right_category]
        )

    left_series = pd.Series(left_values)
    right_series = pd.Series(right_values)
    distance_comparison_rows.append(
        {
            "left": left_name,
            "right": right_name,
            "spearman_correlation": left_series.rank().corr(
                right_series.rank()
            ),
            "mean_absolute_difference": (
                left_series - right_series
            ).abs().mean(),
        }
    )

distance_comparison = pd.DataFrame(distance_comparison_rows)
distance_comparison.to_csv(
    CASE_WORK_DIR / "case-distance-comparison.csv",
    index=False,
)

label_to_category = category_map.set_index("label")["category"].to_dict()
normalized_bytes_by_category = {
    label_to_category[label]: int(size)
    for label, size in normalized_corpus_sizes.items()
}
support_by_category = included_support.set_index("category")[
    "category_report_count"
].to_dict()


def ncd_diagnostics(matrix, bytes_by_category):
    rows = []
    for left, right in combinations(included_categories, 2):
        rows.append(
            {
                "ncd": float(matrix.loc[left, right]),
                "support_gap": abs(
                    log10(support_by_category[left])
                    - log10(support_by_category[right])
                ),
                "byte_gap": abs(
                    bytes_by_category[left] - bytes_by_category[right]
                ),
            }
        )

    diagnostics = pd.DataFrame(rows)
    return {
        "ncd_support_spearman": diagnostics["ncd"].rank().corr(
            diagnostics["support_gap"].rank()
        ),
        "ncd_byte_spearman": diagnostics["ncd"].rank().corr(
            diagnostics["byte_gap"].rank()
        ),
    }


diagnostic_rows = [
    {
        "experiment": "normalized",
        **ncd_diagnostics(
            distance_matrices["normalized"],
            normalized_bytes_by_category,
        ),
    }
]

for name, matrix in distance_matrices.items():
    if name == "normalized":
        continue

    if name == "case-full":
        byte_table = case_category_map.loc[
            case_category_map["regime"] == "case-full"
        ]
    else:
        replicate = name.removeprefix("case-balanced-")
        byte_table = case_category_map.loc[
            (case_category_map["regime"] == "case-balanced")
            & (case_category_map["replicate"] == replicate)
        ]

    diagnostic_rows.append(
        {
            "experiment": name,
            **ncd_diagnostics(
                matrix,
                byte_table.set_index("category")["bytes"].to_dict(),
            ),
        }
    )

ncd_diagnostics_table = pd.DataFrame(diagnostic_rows)
ncd_diagnostics_table.to_csv(
    CASE_WORK_DIR / "case-ncd-diagnostics.csv",
    index=False,
)

membership_rows = []
normalized_membership_table = normalized_membership[
    ["label", "cluster"]
].merge(
    category_map[["label", "category"]],
    on="label",
    validate="one_to_one",
)
normalized_membership_table["experiment"] = "normalized"
membership_rows.extend(normalized_membership_table.to_dict(orient="records"))

for name, data in case_results.items():
    if data["status"] != "completed":
        continue
    membership_table = data["membership"][
        ["label", "cluster"]
    ].merge(
        category_map[["label", "category"]],
        on="label",
        validate="one_to_one",
    )
    membership_table["experiment"] = name
    membership_rows.extend(membership_table.to_dict(orient="records"))

case_membership = pd.DataFrame(membership_rows)
case_membership.to_csv(
    CASE_WORK_DIR / "case-cluster-membership.csv",
    index=False,
)


def same_cluster_pairs(membership_table):
    cluster_by_category = membership_table.set_index("category")[
        "cluster"
    ].to_dict()
    return {
        tuple(sorted((left, right)))
        for left, right in combinations(included_categories, 2)
        if cluster_by_category[left] == cluster_by_category[right]
    }


balanced_memberships = {
    name: same_cluster_pairs(
        case_membership.loc[
            case_membership["experiment"] == name
        ]
    )
    for name in case_membership["experiment"].unique()
    if name.startswith("case-balanced-")
}
all_category_pairs = set(combinations(included_categories, 2))
stability_rows = []
for left_name, right_name in combinations(balanced_memberships, 2):
    left_pairs = balanced_memberships[left_name]
    right_pairs = balanced_memberships[right_name]
    stability_rows.append(
        {
            "left": left_name,
            "right": right_name,
            "pairwise_cluster_agreement": len(
                all_category_pairs - (left_pairs ^ right_pairs)
            ) / len(all_category_pairs),
        }
    )

cluster_stability = pd.DataFrame(stability_rows)
cluster_stability.to_csv(
    CASE_WORK_DIR / "case-balanced-cluster-stability.csv",
    index=False,
)

case_comparison_summary = pd.DataFrame(
    [
        {
            "experiment": name,
            "status": data["status"],
            "tree_available": data["status"] == "completed",
            "tree_path": str(
                data.get("output_dir", "") / "tree.nwk"
                if data.get("output_dir")
                else ""
            ),
        }
        for name, data in case_results.items()
    ]
)
case_comparison_summary.to_csv(
    CASE_WORK_DIR / "case-comparison-summary.csv",
    index=False,
)

print("Correlações entre matrizes:")
display(distance_comparison)
print("Diagnóstico de suporte e tamanho:")
display(ncd_diagnostics_table)
print("Estabilidade das réplicas balanceadas:")
display(cluster_stability)
print("Distribuição das combinações:")
display(
    case_combination_comparison[
        ["category", "replicate", "absolute_share_difference"]
    ].groupby(["category", "replicate"], as_index=False)
    .agg(
        mean_absolute_share_difference=(
            "absolute_share_difference",
            "mean",
        ),
        max_absolute_share_difference=(
            "absolute_share_difference",
            "max",
        ),
    )
    .head(28)
)


if not distance_comparison.empty:
    comparison_names = list(distance_matrices)
    distance_agreement = pd.DataFrame(
        np.nan,
        index=comparison_names,
        columns=comparison_names,
    )
    np.fill_diagonal(distance_agreement.values, 1.0)
    for row in distance_comparison.itertuples(index=False):
        distance_agreement.loc[row.left, row.right] = row.spearman_correlation
        distance_agreement.loc[row.right, row.left] = row.spearman_correlation

    figure, axis = plt.subplots(
        figsize=(max(8, 0.82 * len(comparison_names)), max(7, 0.72 * len(comparison_names))),
    )
    image = axis.imshow(
        distance_agreement.to_numpy(),
        cmap="RdBu_r",
        vmin=-1,
        vmax=1,
        aspect="equal",
    )
    axis.set_xticks(np.arange(len(comparison_names)))
    axis.set_xticklabels(comparison_names, rotation=45, ha="right")
    axis.set_yticks(np.arange(len(comparison_names)))
    axis.set_yticklabels(comparison_names)
    axis.set_title(
        "Agreement between experiment distance matrices",
        loc="left",
        weight="bold",
    )
    axis.set_xlabel("Spearman correlation of pairwise NCD rankings")
    axis.set_ylabel("Experiment")
    figure.colorbar(image, ax=axis, label="Spearman correlation", shrink=0.82)
    figure.tight_layout()
    figure.savefig(
        RESULTS_DIR / "case-distance-agreement.png",
        dpi=180,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.show()
else:
    print("No completed case runs are available for the distance-agreement figure.")

if not cluster_stability.empty:
    stability_plot = cluster_stability.copy()
    stability_plot["comparison"] = (
        stability_plot["left"] + " vs " + stability_plot["right"]
    )
    stability_plot = stability_plot.sort_values("pairwise_cluster_agreement")

    figure, axis = plt.subplots(
        figsize=(12, max(5, 0.42 * len(stability_plot))),
    )
    axis.barh(
        stability_plot["comparison"],
        stability_plot["pairwise_cluster_agreement"],
        color=VISUAL_STYLE["accent_color"],
    )
    axis.set_xlim(0, 1)
    axis.set_xlabel("Pairwise cluster agreement")
    axis.set_title(
        "Stability of balanced-replica cluster assignments",
        loc="left",
        weight="bold",
    )
    axis.grid(axis="x", color=VISUAL_STYLE["grid_color"], linewidth=0.7)
    axis.set_axisbelow(True)
    for position, value in enumerate(stability_plot["pairwise_cluster_agreement"]):
        axis.text(
            min(value + 0.02, 0.98),
            position,
            f"{value:.3f}",
            va="center",
            fontsize=8,
            color=VISUAL_STYLE["text_color"],
        )
    figure.tight_layout()
    figure.savefig(
        RESULTS_DIR / "case-cluster-stability.png",
        dpi=180,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.show()
else:
    print("Fewer than two completed balanced runs are available for stability.")


### 7.4 Critérios de interpretação

case-full mede similaridade contextual junto com prevalência e volume. case-balanced mede similaridade contextual sob suporte igual. Se os dois regimes produzirem partições semelhantes, o agrupamento é menos dependente do volume. Se divergirem, essa divergência é um resultado metodológico.

Clusters só devem ser interpretados quando as réplicas balanceadas forem estáveis, a associação com diferenças de tamanho não dominar e os padrões puderem ser descritos diretamente nas combinações dos campos. Nenhum regime produz tipologias jurídicas, causalidade ou inferência individual.
